# Weighted Lebesgue Spaces: Flat vs Radial Laplacian

The weight $w(r) = r^2$ arises naturally in spherical coordinates: the radial inner product is
$\langle u, v \rangle_{r^2} = \int u(r)\, v(r)\, r^2\, dr$.

The natural operator on this space is the **radial Laplacian**
$-\frac{d^2}{dr^2} - \frac{2}{r}\frac{d}{dr}$,
which is self-adjoint w.r.t. $\langle\cdot,\cdot\rangle_{r^2}$.
Its Dirichlet eigenfunctions are $\varphi_n(r) \propto \sin(n\pi r/R)/r$ — qualitatively
different from the flat $\sin(n\pi r/R)$.

This notebook demonstrates:
1. How `WeightedLebesgue` implements $L^2(r^2)$ via `MassWeightedHilbertSpace`
2. The different eigenfunctions of the flat vs radial Laplacian
3. Sampling from priors using `KLSampler` on both geometries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from intervalinf import IntervalDomain, Function
from intervalinf.spaces import Lebesgue, WeightedLebesgue
from intervalinf.core.boundary import BoundaryConditions
from intervalinf.core.config import IntegrationConfig
from intervalinf.operators import Laplacian, RadialLaplacian, BesselSobolevInverse
from intervalinf.sampling import KLSampler

print("Imports OK")

## 1. Spaces and Operators

In [ ]:
domain = IntervalDomain(0.0, 1.0)
bc = BoundaryConditions.dirichlet()
cfg = IntegrationConfig(method="simpson", n_points=4000)
N = 20

# Plain L²([0,1]) + flat Laplacian  (-d²/dr²)
space_flat = Lebesgue(N, domain, basis=None, integration_config=cfg)
L_flat = Laplacian(space_flat, bc, 1.0, method="spectral", dofs=N, integration_config=cfg)

# Weighted L²([0,1]; r²) + radial Laplacian  (-d²/dr² - 2/r d/dr)
space_radial = WeightedLebesgue(N, domain, lambda r: np.asarray(r)**2, integration_config=cfg)
L_radial = RadialLaplacian(space_radial, bc, 1.0, method="spectral", dofs=N, integration_config=cfg)

print("Flat    eigenvalues (first 3):", [f"{L_flat.get_eigenvalue(j):.3f}"   for j in range(3)])
print("Radial  eigenvalues (first 3):", [f"{L_radial.get_eigenvalue(j):.3f}" for j in range(3)])

## 2. Eigenfunctions: Flat vs Radial Laplacian

Both operators have the same Dirichlet spectrum $\lambda_n = (n\pi)^2$ on $[0,1]$, but
their eigenfunctions are qualitatively different:
- **Flat:** $\varphi_n(r) = \sqrt{2}\,\sin(n\pi r)$ — bounded, symmetric bumps
- **Radial:** $\varphi_n(r) = \sqrt{2}\,\sin(n\pi r)/r$ — peaks sharply near the origin due to the $1/r$ factor

In [ ]:
r = np.linspace(0.01, 0.99, 300)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = ['C0', 'C1', 'C2']
for j in range(3):
    lam_f = L_flat.get_eigenvalue(j)
    lam_r = L_radial.get_eigenvalue(j)
    phi_f = np.array([L_flat.get_eigenfunction(j)(x) for x in r])
    phi_r = np.array([L_radial.get_eigenfunction(j)(x) for x in r])

    axes[0].plot(r, phi_f, color=colors[j], linewidth=2,
                 label=f'φ_{j}  λ={lam_f:.1f}')
    axes[1].plot(r, phi_r, color=colors[j], linewidth=2,
                 label=f'φ_{j}  λ={lam_r:.1f}')
    axes[2].plot(r, phi_r, color=colors[j], linestyle='--', linewidth=2)
    axes[2].plot(r, phi_f, color=colors[j], linewidth=2)

axes[0].set_title('Flat Laplacian on $L^2$\neigenfns $\\sim \\sin(n\\pi r)$')
axes[1].set_title('Radial Laplacian on $L^2(r^2)$\neigenfns $\\sim \\sin(n\\pi r)/r$')
axes[2].set_title('Overlay: flat (solid) vs radial (dashed)')

for ax in axes:
    ax.set_xlabel('r')
    ax.axhline(0, color='k', linewidth=0.5)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Bessel-Sobolev Prior: Flat vs Radial

In [ ]:
k, s = 2.0, 1.0

C_flat   = BesselSobolevInverse(space_flat,   space_flat,   k, s, L_flat,   dofs=N, integration_config=cfg)
C_radial = BesselSobolevInverse(space_radial, space_radial, k, s, L_radial, dofs=N, integration_config=cfg)

print(f"Flat   prior:  fast={C_flat._can_use_fast_transforms}")
print(f"Radial prior:  radial_fast={C_radial._radial_dirichlet_fast}")

## 4. Prior Covariance Kernel

In [ ]:
# Covariance kernel: C(r, r0) = (C δ_{r0})(r), approximated by applying C to a narrow bump at r0.
# Shows how the prior "spreads" information differently in flat vs radial geometry.

r0_values = [0.25, 0.5, 0.75]
r_eval = np.linspace(0.02, 0.98, 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for r0 in r0_values:
    bump = Function(domain, evaluate_callable=lambda x, r0=r0: np.exp(-((np.asarray(x) - r0)**2) / (2*0.02**2)))
    Cf_flat   = C_flat(bump)
    Cf_radial = C_radial(bump)
    fv_flat   = np.array([Cf_flat(x)   for x in r_eval])
    fv_radial = np.array([Cf_radial(x) for x in r_eval])
    axes[0].plot(r_eval, fv_flat,   linewidth=2, label=f'r₀={r0}')
    axes[1].plot(r_eval, fv_radial, linewidth=2, label=f'r₀={r0}')

axes[0].set_title(f'Flat prior  C=(k²–Δ)⁻¹  k={k}, s={s}')
axes[1].set_title(f'Radial prior C=(k²–Δᵣ)⁻¹  k={k}, s={s}')
for ax in axes:
    ax.set_xlabel('r')
    ax.set_ylabel('(C bump)(r)')
    ax.legend()
    ax.axhline(0, color='k', linewidth=0.5)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Sampling from the Prior (KL Expansion)

The `KLSampler` draws samples via the Karhunen-Loève expansion:
$$u = \sum_{j} \sqrt{\lambda_j}\, z_j \varphi_j, \quad z_j \sim \mathcal{N}(0,1)$$

Thanks to the fix in `kl_sampler.py`, it now works for `WeightedLebesgue` domains
(whose eigenfunctions are already w-orthonormal).

In [ ]:
# KLSampler works for both: for WeightedLebesgue the eigenfunctions are already
# w-orthonormal, so no eigenvalue adjustment is needed.
sampler_flat   = KLSampler(L_flat,   n_modes=15)
sampler_radial = KLSampler(L_radial, n_modes=15)

print(f"Flat sampler:   {sampler_flat.n_modes} modes")
print(f"Radial sampler: {sampler_radial.n_modes} modes")

# Draw samples
np.random.seed(42)
n_samples = 8
r_plot = np.linspace(0.01, 0.99, 200)

samples_flat   = sampler_flat.samples(n_samples)
samples_radial = sampler_radial.samples(n_samples)

samples_flat_vals   = np.array([[s(r) for r in r_plot] for s in samples_flat])
samples_radial_vals = np.array([[s(r) for r in r_plot] for s in samples_radial])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for vals in samples_flat_vals:
    axes[0].plot(r_plot, vals, alpha=0.6, linewidth=1.5)
axes[0].set_title(f'Flat Laplacian Samples (n={n_samples}, KLSampler)')
axes[0].set_xlabel('r')
axes[0].grid(True, alpha=0.3)

for vals in samples_radial_vals:
    axes[1].plot(r_plot, vals, alpha=0.6, linewidth=1.5)
axes[1].set_title(f'Radial Laplacian Samples (n={n_samples}, KLSampler)')
axes[1].set_xlabel('r')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

idx_mid = len(r_plot) // 2
print(f"Flat:   std @ r≈{r_plot[idx_mid]:.2f} ≈ {samples_flat_vals[:, idx_mid].std():.4f}")
print(f"Radial: std @ r≈{r_plot[idx_mid]:.2f} ≈ {samples_radial_vals[:, idx_mid].std():.4f}")

## Summary

- **`WeightedLebesgue(r²)`** implements $L^2(r^2)$ via a mass operator — the natural space for spherically symmetric problems
- The **radial Laplacian** is self-adjoint on $L^2(r^2)$; its eigenfunctions are $\sim \sin(n\pi r)/r$, not $\sin(n\pi r)$
- The **flat Laplacian** is self-adjoint on plain $L^2$; using it on a weighted space would break self-adjointness
- **`KLSampler` now supports multiplicative weights**: eigenfunctions are already orthonormal w.r.t. the weighted inner product, so no adjustment is needed